# TreeSearchPlayer.fallback() をオーバーライドして、探索できない局面での代替方策をカスタマイズする方法を示す

fallback() が使われる2つの局面と既定の挙動は docs/api/README.md の
TreeSearchPlayer「オーバーライド可能なフック」節を参照。ここでは「攻撃技があれば
優先する」という軽量なヒューリスティックに差し替える最小例を示す。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player
from jpoke.enums import Command
from jpoke.players.tree_search_player import TreeSearchPlayer

In [ ]:
class AttackPreferringFallbackPlayer(TreeSearchPlayer):
    """fallback時（探索できない局面）は攻撃技を優先するAI（fallback()の拡張例）。"""

    def fallback(self, battle: Battle) -> Command:
        commands = battle.get_available_commands(self)

        def is_attack(command: Command) -> bool:
            # is_regular_move の定義は docs/api/README.md の Command 節を参照
            # （変化技〔まもる等〕も含む点に注意。攻撃技かどうかはmove.is_attackで判定する）
            return command.is_regular_move and battle.command_to_move(self, command).is_attack

        attack_commands = [c for c in commands if is_attack(c)]
        if attack_commands:
            return attack_commands[0]
        # 攻撃技がない場合は既定のランダム選択に委譲する
        return super().fallback(battle)

In [ ]:
ai_player = AttackPreferringFallbackPlayer("FallbackAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("カビゴン", move_names=["まもる", "のしかかり"])

opponent_player = Player("Opponent")
opponent_player.add_pokemon("カビゴン", move_names=["のしかかり"])

In [ ]:
battle = Battle(ai_player, opponent_player, seed=1)
battle.start()

In [ ]:
# 1ターン目: 相手の技が未公開でestimate_opponentも未実装のためfallback()に
# 委譲される。既定のランダム選択なら「まもる」も等確率で選ばれ得るが、この
# 実装では攻撃技「のしかかり」が優先して選ばれる
battle.step()
active = battle.get_active(ai_player)
print(f"1ターン目にFallbackAIが選んだ技: {active.last_move.name if active.last_move else None}")
battle.print_logs()

試してみよう: fallback() を既定実装のまま（オーバーライドせず）
何度か実行すると、1ターン目の選択が「まもる」「のしかかり」でばらつくのに対し、
このサンプルでは常に「のしかかり」が選ばれ続けることを比較できる